# Resolver calibration and real decision examples

Run in the **new `active-learning` repository**, using the `al-pipeline` environment.
This notebook reruns the current `AppleCiDErCatalogResolver` on saved real evidence,
inspects `LabelResolution`, and verifies its SQLite/JSON history representation.

**CPU only, offline.** It does not fetch catalogs, load model weights, train models,
or alter experiment pools or existing history databases. Every execution creates
a separate report directory. Full reference evidence is embedded in this notebook's
metadata, so the downloaded notebook is self-contained for decision examples.

Two populations remain separate:

1. Historical calibration: 180 objects sampled at 20 per original historical subtype.
   Delta's `calibration_objects.csv` supplies independent reference labels.
2. One real entropy-acquired object: `ZTF25aafiaod`, used only to illustrate an
   outside-taxonomy decision. It is **not** added to the calibration denominator.

Catalog information is retrospective. These are not measurements of label availability
at the original acquisition time. The calibration sample is not an estimate of the
label yield in a new, unlabeled alert stream.

## 1. Paths and imports

Save this file as `notebooks/01_resolver_calibration_and_examples.ipynb` in the new repo.
Keep the existing Delta paths. Optional environment overrides are shown below.
The default uses the embedded successful evidence snapshot; set
`AL_RESOLVER_EVIDENCE` only to deliberately use a different saved `EvidenceBatch`.

In [1]:
from pathlib import Path
from collections import Counter
from collections.abc import Mapping
from datetime import datetime, timezone
from dataclasses import asdict, is_dataclass
from enum import Enum
import hashlib
import inspect
import json
import os
import sqlite3
import subprocess
import sys
import uuid

import pandas as pd
try:
    from IPython.display import display, Markdown
except ImportError:  # Also allow execution as ordinary Python on Delta.
    def display(value):
        print(value)
    def Markdown(value):
        return value

def find_repo():
    configured = os.environ.get("AL_PIPELINE_REPO")
    candidates = [Path(configured)] if configured else [Path.cwd(), *Path.cwd().parents]
    for path in candidates:
        if (path / "src/al_pipeline").is_dir():
            return path.resolve()
    raise FileNotFoundError("Run inside active-learning, or set AL_PIPELINE_REPO.")

REPO_ROOT = find_repo()
sys.path.insert(0, str(REPO_ROOT / "src"))
import al_pipeline
assert Path(al_pipeline.__file__).resolve().is_relative_to(REPO_ROOT / "src"), (
    "A different al_pipeline was already imported. Restart the kernel."
)
from al_pipeline.core.evidence.types import Evidence, EvidenceBatch, EvidenceFailure
from al_pipeline.core.oracle.resolution import LabelResolution, LabelResolutionBatch, LabelStatus
from al_pipeline.core.state.history import SQLiteObjectHistoryStore
from al_pipeline.astronomy.labels.applecider_catalog import AppleCiDErCatalogResolver

NAME = "01_resolver_calibration_and_examples.ipynb"
configured_nb = os.environ.get("AL_RESOLVER_NOTEBOOK")
nb_candidates = [Path(configured_nb)] if configured_nb else [
    Path.cwd() / NAME, REPO_ROOT / "notebooks" / NAME, REPO_ROOT / NAME,
]
NOTEBOOK_PATH = next((p.resolve() for p in nb_candidates if p.is_file()), None)
if NOTEBOOK_PATH is None:
    raise FileNotFoundError("Set AL_RESOLVER_NOTEBOOK to this notebook's full path.")
SNAPSHOT = json.loads(NOTEBOOK_PATH.read_text())["metadata"]["resolver_audit_snapshot"]

CALIBRATION_DIR = Path(os.environ.get("AL_RESOLVER_CALIBRATION_DIR",
    "/work/hdd/bcrv/dkourtesis/applecider_adaptive/label_audits/"
    "historical_label_boom_resolver_calibration_v1"))
TRUTH_PATH = Path(os.environ.get("AL_RESOLVER_TRUTH", str(CALIBRATION_DIR / "calibration_objects.csv")))
EVIDENCE_PATH = os.environ.get("AL_RESOLVER_EVIDENCE")  # None means embedded snapshot.
# Set True on Delta if the independent reference CSV must be present.
REQUIRE_TRUTH = os.environ.get("AL_RESOLVER_REQUIRE_TRUTH", "0") == "1"
OUTPUT_ROOT = Path(os.environ.get("AL_RESOLVER_OUTPUT",
    "/projects/bcrv/dkourtesis/applecider_adaptive/reports/al_pipeline_resolver_audit_v1"))
RUN_ID = "resolver_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ_") + uuid.uuid4().hex[:8]
OUTPUT = OUTPUT_ROOT / RUN_ID
OUTPUT.mkdir(parents=True, exist_ok=False)
print("Package:", al_pipeline.__file__)
print("Report:", OUTPUT)
print("Reference CSV:", TRUTH_PATH, "exists:", TRUTH_PATH.is_file())

Package: /workspace/scratch/426547cfbd99/extraction/al-pipeline/src/al_pipeline/__init__.py
Report: /workspace/scratch/426547cfbd99/resolver-audit/validation_reports/resolver_20261008T123243Z_5eb09ca2
Reference CSV: /work/hdd/bcrv/dkourtesis/applecider_adaptive/label_audits/historical_label_boom_resolver_calibration_v1/calibration_objects.csv exists: False


## 2. Load saved evidence without changing it

`Evidence` records a provider's native information. It is weaker than a resolved
label. `EvidenceFailure` records a failed request; a failed query is never treated
as a successful query with no matches. `requested_ids` keeps objects that had no
evidence, so they still receive a decision.

The acquired example retains its original VSX catalog record. Its separation
comes from the saved table (six decimal places); no values or labels are invented.
Its evidence coverage is narrower than the historical calibration's five catalogs.

In [2]:
def plain(value):
    if isinstance(value, Enum):
        return value.value
    if isinstance(value, Mapping):
        return {str(k): plain(v) for k, v in value.items()}
    if isinstance(value, (tuple, list)):
        return [plain(v) for v in value]
    if isinstance(value, (set, frozenset)):
        return [plain(v) for v in sorted(value, key=str)]
    if is_dataclass(value):
        return {name: plain(getattr(value, name)) for name in value.__dataclass_fields__}
    return value

def write_json(path, value):
    path.write_text(json.dumps(plain(value), indent=2, sort_keys=True, allow_nan=False) + "\n")

def digest(value):
    return hashlib.sha256(json.dumps(plain(value), sort_keys=True, allow_nan=False).encode()).hexdigest()

def load_batch(payload):
    assert set(payload) == {"requested_ids", "evidence", "failures"}, "Unexpected cache schema."
    return EvidenceBatch(
        requested_ids=payload["requested_ids"],
        evidence=[Evidence(**item) for item in payload["evidence"]],
        failures=[EvidenceFailure(**item) for item in payload["failures"]],
    )

historical_payload = json.loads(Path(EVIDENCE_PATH).read_text()) if EVIDENCE_PATH else SNAPSHOT["historical_evidence"]
historical = load_batch(historical_payload)
acquired_payload = SNAPSHOT["acquired_example_evidence"]
acquired = load_batch(acquired_payload)
assert set(historical.requested_ids).isdisjoint(acquired.requested_ids)
write_json(OUTPUT / "historical_evidence.json", historical_payload)
write_json(OUTPUT / "acquired_example_evidence.json", acquired_payload)
print("Historical objects / evidence / failures:", len(historical.requested_ids), len(historical.evidence), len(historical.failures))

source_rows = []
for name, batch in [("historical", historical), ("acquired_example", acquired)]:
    for source in sorted({e.source for e in batch.evidence} | {e.source for e in batch.failures}):
        source_rows.append({
            "population": name, "source": source,
            "matched_objects": len({e.object_id for e in batch.evidence if e.source == source}),
            "evidence_rows": sum(e.source == source for e in batch.evidence),
            "failed_objects": len({e.object_id for e in batch.failures if e.source == source}),
            "failure_rows": sum(e.source == source for e in batch.failures),
        })
source_table = pd.DataFrame(source_rows)
display(source_table)
source_table.to_csv(OUTPUT / "evidence_by_source.csv", index=False)

Historical objects / evidence / failures: 180 284 0
         population        source  ...  failed_objects  failure_rows
0        historical      DESI_DR1  ...               0             0
1        historical           NED  ...               0             0
2        historical           TNS  ...               0             0
3        historical           VSX  ...               0             0
4        historical  milliquas_v8  ...               0             0
5  acquired_example           VSX  ...               0             0

[6 rows x 6 columns]


## 3. Apply the current astronomy rules through the general resolver

No historical reference labels are passed to the resolver. Rules generate claims
from native evidence. Agreeing claims are combined; conflicting claims remain
unresolved. An unsupported catalog type remains unresolved unless an explicit
outside-taxonomy rule supports that decision.

Record the actual rule settings and source hashes so this report remains interpretable
if policy changes later. This notebook does not change thresholds or mappings.

In [3]:
resolver = AppleCiDErCatalogResolver()
historical_results = resolver.resolve(historical)
acquired_results = resolver.resolve(acquired)
assert [r.object_id for r in historical_results.resolutions] == list(historical.requested_ids)

policy = [{"rule": type(rule).__name__, "settings": plain(asdict(rule)) if is_dataclass(rule) else repr(rule)} for rule in resolver.rules]
source_hashes = {}
for cls in [AppleCiDErCatalogResolver, *type(resolver).__mro__[1:-1],
            *[type(rule) for rule in resolver.rules], Evidence, LabelResolution, SQLiteObjectHistoryStore]:
    path = Path(inspect.getfile(cls)).resolve()
    source_hashes[str(path.relative_to(REPO_ROOT))] = hashlib.sha256(path.read_bytes()).hexdigest()
try:
    commit = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], stderr=subprocess.DEVNULL, text=True).strip()
except (subprocess.CalledProcessError, FileNotFoundError):
    commit = None

manifest = {"run_id": RUN_ID, "package_path": str(Path(al_pipeline.__file__).resolve()),
    "git_commit": commit, "source_sha256": source_hashes, "rules": policy,
    "historical_evidence_sha256": digest(historical_payload),
    "acquired_evidence_sha256": digest(acquired_payload),
    "source_notes": SNAPSHOT["source_notes"], "evidence_override": EVIDENCE_PATH}
write_json(OUTPUT / "policy_manifest.json", manifest)

rows = []
for population, batch, results in [("historical", historical, historical_results), ("acquired_example", acquired, acquired_results)]:
    for r in results.resolutions:
        assert (r.label is not None) == (r.status is LabelStatus.LABELED)
        rows.append({"population": population, "object_id": r.object_id, "status": r.status.value,
            "label": r.label, "source": r.source,
            "native_label_json": json.dumps(plain(r.native_label)),
            "reason": (r.provenance or {}).get("reason"),
            "evidence_count": len(batch.for_object(r.object_id)),
            "failure_count": len(batch.failures_for_object(r.object_id)),
            "provenance_json": json.dumps(plain(r.provenance), sort_keys=True)})
decisions = pd.DataFrame(rows)
decisions.to_csv(OUTPUT / "resolution_records.csv", index=False)
display(pd.crosstab(decisions.population, decisions.status))
write_json(OUTPUT / "resolution_records.json", {
    "historical": [plain(r) for r in historical_results.resolutions],
    "acquired_example": [plain(r) for r in acquired_results.resolutions]})

status            labeled  outside_label_space  unresolved
population                                                
acquired_example        0                    1           0
historical            166                    0          14


## 4. Check independent historical labels, when available

On Delta the original audit directory should contain `calibration_objects.csv`.
Only its independent `object_id`, `original_label`, and `target_label` columns
are used. Missing, duplicate, mismatched, or inconsistent reference rows fail
the check. Reference labels are not inferred from the order of the evidence cache.

Count labeled matches, labeled mismatches, outside-taxonomy contradictions, and
unresolved outcomes separately. Query failures are also reported separately.
If the reference file is missing, **accuracy is not measured**; only decision counts
and storage checks are valid. Set `REQUIRE_TRUTH = True` above to require calibration.

In [4]:
HISTORICAL_TO_TARGET = {
    "AGN": "AGN", "Cataclysmic": "CV", "SN Ia": "SN I", "SN Ib": "SN I", "SN Ic": "SN I",
    "SN II": "SN II", "SN IIP": "SN II", "SN IIp": "SN II", "SN IIn": "SN II",
    "Tidal Disruption Event": "TDE", "TDE": "TDE",
}
calibration_summary = {"accuracy_checked": False, "reference_csv": str(TRUTH_PATH)}
if TRUTH_PATH.is_file():
    truth = pd.read_csv(TRUTH_PATH, dtype={"object_id": str, "original_label": str, "target_label": str})
    required = {"object_id", "original_label", "target_label"}
    assert required.issubset(truth.columns), f"Reference CSV needs {required}"
    truth = truth[["object_id", "original_label", "target_label"]].copy()
    assert not truth.isna().any().any() and not truth.object_id.duplicated().any()
    assert set(truth.object_id) == set(historical.requested_ids), "Reference IDs differ from the evidence cache."
    mapped = truth.original_label.map(HISTORICAL_TO_TARGET)
    assert mapped.notna().all() and mapped.eq(truth.target_label).all(), "Historical mapping mismatch."
    if not EVIDENCE_PATH:
        assert len(truth) == 180 and truth.original_label.nunique() == 9
        assert truth.groupby("original_label").size().eq(20).all()
    calibration = truth.merge(decisions[decisions.population.eq("historical")], on="object_id", validate="one_to_one")
    calibration["correct_labeled"] = calibration.status.eq("labeled") & calibration.label.eq(calibration.target_label)
    calibration["unsafe_labeled_mismatch"] = calibration.status.eq("labeled") & ~calibration.label.eq(calibration.target_label)
    calibration["outside_contradiction"] = calibration.status.eq("outside_label_space")
    calibration_summary.update({"accuracy_checked": True, "reference_sha256": hashlib.sha256(TRUTH_PATH.read_bytes()).hexdigest(),
        "objects": len(calibration), "correct_labeled": int(calibration.correct_labeled.sum()),
        "unsafe_labeled_mismatches": int(calibration.unsafe_labeled_mismatch.sum()),
        "outside_contradictions": int(calibration.outside_contradiction.sum()),
        "unresolved": int(calibration.status.eq("unresolved").sum()),
        "objects_with_query_failures": int(calibration.failure_count.gt(0).sum())})
    calibration.to_csv(OUTPUT / "historical_calibration.csv", index=False)
    per_subtype = calibration.groupby(["original_label", "target_label"]).agg(
        objects=("object_id", "size"), correct_labeled=("correct_labeled", "sum"),
        unsafe_mismatches=("unsafe_labeled_mismatch", "sum"),
        outside_contradictions=("outside_contradiction", "sum"))
    per_subtype["unresolved"] = calibration.groupby(["original_label", "target_label"]).status.apply(lambda s: s.eq("unresolved").sum())
    display(per_subtype)
    per_subtype.to_csv(OUTPUT / "calibration_by_subtype.csv")
    unsafe = calibration[calibration.unsafe_labeled_mismatch | calibration.outside_contradiction]
    unsafe.to_csv(OUTPUT / "calibration_disagreements.csv", index=False)
    display(unsafe) if not unsafe.empty else print("No disagreements with historical reference labels.")
else:
    if REQUIRE_TRUTH:
        raise FileNotFoundError(TRUTH_PATH)
    display(Markdown("**Reference CSV unavailable: no calibration accuracy claim is made.**"))
write_json(OUTPUT / "calibration_summary.json", calibration_summary)
print(json.dumps(calibration_summary, indent=2))

**Reference CSV unavailable: no calibration accuracy claim is made.**
{
  "accuracy_checked": false,
  "reference_csv": "/work/hdd/bcrv/dkourtesis/applecider_adaptive/label_audits/historical_label_boom_resolver_calibration_v1/calibration_objects.csv"
}


## 5. Inspect three real `LabelResolution` records

Each record contains `object_id`, `status`, `label`, `source`, `native_label`, and
`provenance`. Only **labeled** records contain a target `label`. For an unresolved
object the top-level `source` and `native_label` may be null: inspect its retained
`EvidenceBatch` to see the actual retrieved information.

The selected examples are fixed real objects, with assertions against the current
outcomes. If policy changes their decisions, stop and revise the slide rather than
silently substituting an invented example.

In [5]:
by_id = {r.object_id: r for r in (*historical_results.resolutions, *acquired_results.resolutions)}
EXAMPLE_IDS = {"labeled": "ZTF18abklaoj", "unresolved": "ZTF18aaseyca", "outside_label_space": "ZTF25aafiaod"}
examples = {}
for status, object_id in EXAMPLE_IDS.items():
    result = by_id[object_id]
    assert result.status.value == status, f"Policy changed {object_id}: revise this example."
    batch = historical if object_id in historical.requested_ids else acquired
    examples[status] = {"population": "historical" if batch is historical else "acquired_example",
        "resolution": plain(result), "evidence": [plain(e) for e in batch.for_object(object_id)],
        "failures": [plain(f) for f in batch.failures_for_object(object_id)]}
    display(Markdown(f"### {status}: `{object_id}`"))
    print("LabelResolution:")
    print(json.dumps(examples[status]["resolution"], indent=2))
    print("Retrieved evidence:")
    display(pd.DataFrame([{"source": e.source, "kind": e.kind, "value": plain(e.value),
        "separation_arcsec": e.provenance.get("separation_arcsec")} for e in batch.for_object(object_id)]))
    print("Failures:", examples[status]["failures"])
write_json(OUTPUT / "resolver_examples.json", examples)

### labeled: `ZTF18abklaoj`
LabelResolution:
{
  "object_id": "ZTF18abklaoj",
  "status": "labeled",
  "label": "SN I",
  "source": "TNS",
  "native_label": "SN IA",
  "provenance": {
    "reason": "resolving_evidence",
    "claims": [
      {
        "status": "labeled",
        "label": "SN I",
        "source": "TNS",
        "native_label": "SN IA",
        "rule": "TNSLabelRule",
        "provenance": {
          "separation_arcsec": 0.17451901875820358,
          "catalog_id": 28226,
          "tns_name": "2018edd",
          "tns_name_prefix": "SN",
          "exact_internal_name_match": true
        }
      }
    ],
    "evidence_count": 1,
    "failures": []
  }
}
Retrieved evidence:
  source  ... separation_arcsec
0    TNS  ...          0.174519

[1 rows x 4 columns]
Failures: []
### unresolved: `ZTF18aaseyca`
LabelResolution:
{
  "object_id": "ZTF18aaseyca",
  "status": "unresolved",
  "label": null,
  "source": null,
  "native_label": null,
  "provenance": {
    "reason": "

## 6. SQLite history stores JSON fields, and reconstructs Python events

Use a **new audit-local database**, not an experiment's history. Persisting these
resolution events does not acquire objects, modify a pool, or train a model.
`SQLiteObjectHistoryStore` is the repository's existing persistence machinery.

The SQL table stores `status` and `source` as text, and `label_json`,
`native_label_json`, and `provenance_json` as JSON text. The API reads that JSON
back into `ResolutionEvent` objects, with timestamp, run ID and cycle alongside it.
The audit has no real AL cycle, so `cycle` is left null. Full input evidence is
retained separately in the exported cache and examples JSON; it is not all copied
into every SQLite resolution row.

In [6]:
DB_PATH = OUTPUT / "resolution_history.sqlite"
assert not DB_PATH.exists(), "Rerun from the paths cell to create a fresh audit."
history = SQLiteObjectHistoryStore(DB_PATH)
history.record_resolutions(historical_results, run_id=RUN_ID)
history.record_resolutions(acquired_results, run_id=RUN_ID)
events = history.resolution_events()
original = [*historical_results.resolutions, *acquired_results.resolutions]
assert len(events) == len(original)
for result, event in zip(original, events):
    for field in ("object_id", "status", "label", "source", "native_label"):
        assert plain(getattr(event, field)) == plain(getattr(result, field)), (result.object_id, field)
    assert plain(event.provenance) == plain(result.provenance or {})
    assert event.run_id == RUN_ID and event.cycle is None
print(f"PASS: {len(events)} resolution events round-trip through SQLite.")

with sqlite3.connect(DB_PATH) as connection:
    sql_examples = pd.read_sql_query(
        "SELECT object_id, status, label_json, source, native_label_json, provenance_json "
        "FROM resolution_events WHERE object_id IN (?, ?, ?) ORDER BY id",
        connection, params=list(EXAMPLE_IDS.values()))
display(sql_examples)
for row in sql_examples.itertuples(index=False):
    r = by_id[row.object_id]
    assert json.loads(row.label_json) == plain(r.label)
    assert json.loads(row.native_label_json) == plain(r.native_label)
    assert json.loads(row.provenance_json) == plain(r.provenance or {})
sql_examples.to_csv(OUTPUT / "example_sqlite_rows.csv", index=False)
storage_check = {"passed": True, "events": len(events), "example_rows": len(sql_examples), "database": str(DB_PATH)}
write_json(OUTPUT / "storage_check.json", storage_check)

PASS: 181 resolution events round-trip through SQLite.
      object_id  ...                                    provenance_json
0  ZTF18aaseyca  ...  {"evidence_count":1,"evidence_sources":["NED"]...
1  ZTF18abklaoj  ...  {"claims":[{"label":"SN I","native_label":"SN ...
2  ZTF25aafiaod  ...  {"claims":[{"label":null,"native_label":"RRAB"...

[3 rows x 6 columns]


## 7. Export the next presentation slide from these decisions

The frame is designed for the existing light Beamer theme. Paste it immediately
after **A resolver turns evidence into a decision**, or use `\input{resolver_examples_slide.tex}`.
The slide deliberately shows only the decisive evidence and outcome; the notebook
and JSON export retain the complete dictionaries and policy provenance.

In [7]:
def tex(value):
    replacements = {"\\": r"\textbackslash{}", "_": r"\_", "%": r"\%", "&": r"\&", "#": r"\#", "$": r"\$", "{": r"\{", "}": r"\}"}
    return "".join(replacements.get(c, c) for c in str(value))

lab = examples["labeled"]["resolution"]
unres = examples["unresolved"]["resolution"]
outside = examples["outside_label_space"]["resolution"]
tns = next(e for e in examples["labeled"]["evidence"] if e["source"] == "TNS")
ned = next(e for e in examples["unresolved"]["evidence"] if e["source"] == "NED")
vsx = examples["outside_label_space"]["evidence"][0]
assert any(c["rule"] == "TNSLabelRule" for c in lab["provenance"]["claims"])
assert ned["value"]["objtype"] == "G" and not examples["unresolved"]["failures"]
assert any(c["rule"] == "VSXOutsideRule" for c in outside["provenance"]["claims"])

slide = r"""% Generated from real cached evidence by 01_resolver_calibration_and_examples.ipynb.
% Historical examples and the acquired example are separate populations.
% Catalog evidence is retrospective; the acquired example covers four catalogs.
\begin{frame}{What does a resolver decision look like?}
  {\small \texttt{LabelResolution}: a decision and its provenance for one object.}\par
  \vspace{2mm}
  {\small
  \setlength{\tabcolsep}{4pt}
  \renewcommand{\arraystretch}{1.15}
  \begin{tabular}{@{}>{\raggedright\arraybackslash}p{3.6cm}>{\raggedright\arraybackslash}p{4.25cm}>{\raggedright\arraybackslash}p{4.6cm}@{}}
    \toprule
    \textbf{Real object} & \textbf{Retrieved evidence} & \textbf{Decision} \\
    \midrule
    \texttt{@LAB_ID@} & TNS: \texttt{@NATIVE@}\newline
      Exact ID; @TNS_SEP@$''$ & \textbf{Labeled}\newline
      \texttt{label = "@LABEL@"} \\[2mm]
    \texttt{@UNRES_ID@} & NED: \texttt{G} (galaxy)\newline
      Context only; @NED_SEP@$''$ & \textbf{Unresolved}\newline
      \texttt{label = null} \\[2mm]
    \texttt{@OUT_ID@} & VSX: \texttt{@OUT_NATIVE@}\newline
      Match: @VSX_SEP@$''$ & \textbf{Outside the taxonomy}\newline
      \texttt{label = null} \\
    \bottomrule
  \end{tabular}\par}
  \vspace{3mm}
  {\small \textbf{Python record} $\longrightarrow$ \textbf{SQLite history}\par
  Label, native label and provenance are stored as JSON fields.}\par
  \takeaway{A missing target label can mean uncertainty or a known outside class.}
  \plan{S04b}{Confirm calibration accuracy on Delta using the reference CSV.}
\end{frame}
"""
values = {"@LAB_ID@": tex(lab["object_id"]), "@NATIVE@": tex(tns["value"]["type"]),
    "@LABEL@": tex(lab["label"]), "@TNS_SEP@": f'{tns["provenance"]["separation_arcsec"]:.3f}',
    "@UNRES_ID@": tex(unres["object_id"]), "@NED_SEP@": f'{ned["provenance"]["separation_arcsec"]:.3f}',
    "@OUT_ID@": tex(outside["object_id"]), "@OUT_NATIVE@": tex(outside["native_label"]),
    "@VSX_SEP@": f'{vsx["provenance"]["separation_arcsec"]:.3f}'}
for token, value in values.items():
    slide = slide.replace(token, value)
(OUTPUT / "resolver_examples_slide.tex").write_text(slide)
print(slide)

% Generated from real cached evidence by 01_resolver_calibration_and_examples.ipynb.
% Historical examples and the acquired example are separate populations.
% Catalog evidence is retrospective; the acquired example covers four catalogs.
\begin{frame}{What does a resolver decision look like?}
  {\small \texttt{LabelResolution}: a decision and its provenance for one object.}\par
  \vspace{2mm}
  {\small
  \setlength{\tabcolsep}{4pt}
  \renewcommand{\arraystretch}{1.15}
  \begin{tabular}{@{}>{\raggedright\arraybackslash}p{3.6cm}>{\raggedright\arraybackslash}p{4.25cm}>{\raggedright\arraybackslash}p{4.6cm}@{}}
    \toprule
    \textbf{Real object} & \textbf{Retrieved evidence} & \textbf{Decision} \\
    \midrule
    \texttt{ZTF18abklaoj} & TNS: \texttt{SN Ia}\newline
      Exact ID; 0.175$''$ & \textbf{Labeled}\newline
      \texttt{label = "SN I"} \\m]
    \texttt{ZTF18aaseyca} & NED: \texttt{G} (galaxy)\newline
      Context only; 0.468$''$ & \textbf{Unresolved}\newline
      \texttt{lab

## 8. Interpretation and remaining work

- The historical decision counts describe this saved 180-object calibration set.
  Accuracy requires the independent reference CSV; compare `calibration_summary.json`.
- `ZTF18aaseyca` shows how an informative galaxy match can remain unresolved.
  An object with a query failure must be distinguished from this failure-free case.
- `ZTF25aafiaod` demonstrates an explicit VSX outside rule. It does not by itself
  validate that rule on a representative sample of acquired objects. Association
  quality and scientific rule validation still matter.
- An outside decision does not dictate pool handling. Exclude, retain for anomaly
  review, and keep pending are separate workflow policy choices.
- To measure **real-stream automatic labeling yield**, rerun the same resolver on
  full saved acquired batches, retain all query failures, and report their own
  denominator. Do not substitute historical calibration yield for that measurement.
- No Fink adapter or Fink output is used here. No model or broker code is modified.

Exports: policy/input hashes, full evidence, all resolutions, source coverage,
optional calibration tables, three complete example records, SQLite database,
storage checks, and the generated Beamer frame.

In [8]:
summary = {"run_id": RUN_ID, "historical_objects": len(historical.requested_ids),
    "historical_decisions": dict(Counter(r.status.value for r in historical_results.resolutions)),
    "historical_failed_objects": len(historical.objects_with_failures),
    "acquired_example_decisions": dict(Counter(r.status.value for r in acquired_results.resolutions)),
    "calibration": calibration_summary, "storage": storage_check,
    "output": str(OUTPUT), "files": sorted(p.name for p in OUTPUT.iterdir())}
write_json(OUTPUT / "audit_summary.json", summary)
print(json.dumps(summary, indent=2))

{
  "run_id": "resolver_20261008T123243Z_5eb09ca2",
  "historical_objects": 180,
  "historical_decisions": {
    "labeled": 166,
    "unresolved": 14
  },
  "historical_failed_objects": 0,
  "acquired_example_decisions": {
    "outside_label_space": 1
  },
  "calibration": {
    "accuracy_checked": false,
    "reference_csv": "/work/hdd/bcrv/dkourtesis/applecider_adaptive/label_audits/historical_label_boom_resolver_calibration_v1/calibration_objects.csv"
  },
  "storage": {
    "passed": true,
    "events": 181,
    "example_rows": 3,
    "database": "/workspace/scratch/426547cfbd99/resolver-audit/validation_reports/resolver_20261008T123243Z_5eb09ca2/resolution_history.sqlite"
  },
  "output": "/workspace/scratch/426547cfbd99/resolver-audit/validation_reports/resolver_20261008T123243Z_5eb09ca2",
  "files": [
    "acquired_example_evidence.json",
    "calibration_summary.json",
    "evidence_by_source.csv",
    "example_sqlite_rows.csv",
    "historical_evidence.json",
    "policy_manif